# DEEP Week 8 Practical
## Autoregressive language modelling and zero-shot CLIP

In this practical you will build and train a basic autoregressive language model from scratch and use it to generate new text. You will then use a pretrained vision and language model (CLIP) to do zero-shot classification.

*This notebook runs top to bottom in Google Colab or a local Jupyter install. Cells marked **To do** contain placeholders for you to complete; they run as provided but produce nothing useful until you fill them in.*

In [ ]:
import os
import re
import math
import random
import string
import zipfile
import urllib.request

import numpy as np
import torch
import torch.nn as nn


def download(url):
    """Download a file into the current folder (skips if it already exists)."""
    fname = url.split('/')[-1]
    if not os.path.exists(fname):
        urllib.request.urlretrieve(url, fname)
    return fname

We will use a very simple recurrent neural network to learn a language model. This is a small neural network $y_t,h_t=f(x_t,h_{t-1})$ that takes as input the previous hidden state $h_{t-1}$ and the current input character $x_t$ and outputs the next hidden state $h_t$ and scores for all possible next characters $y_t$.

Start by reading carefully through the notes and code to define and train the model. Ask questions if you don't understand part of it.

The following model can write the works of Shakespeare! (Well, nearly).

In [ ]:
class RNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super(RNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.linear_h = nn.Linear(hidden_size, hidden_size)
        self.linear_y = nn.Linear(hidden_size, vocab_size)
        self.tanh = nn.Tanh()

    def forward(self, x, hprev):
        h = self.tanh(self.embedding(x) + self.linear_h(hprev))
        y = self.linear_y(h)
        return h, y

Now we need code to compute a loss to train it. We will provide the model, a sequence of inputs, a sequence of desired outputs (these are just the same as the inputs but lagged by one character), the starting hidden state and the vocabulary size. This code loops over the inputs one character at a time, passing them into the model and updating the hidden state each time. It computes cross entropy loss between the output scores and target labels and returns this along with the final hidden state (which will be used as the initial hidden state for the next sequence).

In [ ]:
def calculate_loss(model, inputs, targets, hprev, vocab_size):
    loss_func = nn.CrossEntropyLoss()
    seq_length = len(inputs)
    outputs = torch.zeros(0, vocab_size)
    for t in range(seq_length):
        # For each character in the input sequence, pass through RNN with previous hidden state
        hprev, y = model(torch.tensor([inputs[t]]), hprev)
        # Gradually build up matrix of output logits of size seq_length * vocab_size
        outputs = torch.cat((outputs, y))

    # Compute cross entropy loss for seq_length actual targets against estimated distributions
    loss = loss_func(outputs, torch.tensor(targets))

    # For truncated backprop, the next subsequence will use the final hidden state
    # but will not backprop through it so we need to detach
    hprev = hprev.detach()

    return loss, hprev

Now we need a dataset. Let's use the complete works of William Shakespeare.

In [ ]:
fname = download('https://www-users.york.ac.uk/~waps101/PADL/shakespeare.txt')
# You can try the complete Sherlock Holmes later too:
# fname = download('https://www-users.york.ac.uk/~waps101/PADL/sherlock.txt')
data = open(fname, 'r', encoding='utf-8', errors='replace').read()  # plain text file
chars = sorted(set(data))
data_size, vocab_size = len(data), len(chars)
print('Data has %d characters, %d unique.' % (data_size, vocab_size))
char_to_ix = {ch: i for i, ch in enumerate(chars)}
ix_to_char = {i: ch for i, ch in enumerate(chars)}

Let's define some hyperparameters

In [ ]:
hidden_size = 100  # size of hidden layer of neurons
seq_length = 25   # number of steps to unroll the RNN for
lr = 0.001
iterations = 5000

Our model is actually quite small

In [ ]:
model = RNN(vocab_size, hidden_size)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(total_params)

We're now ready to train our model. We work our way through the training text, extracting the next subsequence of characters and putting this through the model to get a loss. We don't need to train for long (much less than one epoch) in order to get a reasonable model.

In [ ]:
optim = torch.optim.Adam(model.parameters(), lr=lr)

n, p = 0, 0
while n <= iterations:
    # One iteration of truncated backprop
    if p + seq_length + 1 >= len(data) or n == 0:
        p = 0
        hprev = torch.zeros(hidden_size,)  # reset RNN memory
    # Extract next subsequence of characters
    inputs = [char_to_ix[ch] for ch in data[p:p + seq_length]]
    targets = [char_to_ix[ch] for ch in data[p + 1:p + seq_length + 1]]

    # Compute loss for current subsequence
    loss, hprev = calculate_loss(model, inputs, targets, hprev, vocab_size)
    optim.zero_grad()
    loss.backward()
    optim.step()

    if n % 500 == 0:
        print('[{:}] Loss: {:.2f}'.format(n, loss.item()))

    p += seq_length  # move data pointer
    n += 1           # iteration counter

Ok, so the loss went down but we don't really know what the model has learnt...

**To do**: create a function to sample from the model (i.e. create random passages of text). The function should take as input:

1. The model itself
2. The previous hidden state (which might be all zeros)
3. A seed initial input (i.e. a random integer between 0 and `vocab_size-1`)
4. The desired sequence length

Your function should pass the current input and previous hidden state to the RNN to get the next hidden state and the output. Apply softmax to the outputs to get probabilities over the vocabulary. Now, randomly sample a character from the distribution (hint: `torch.multinomial` will do this). This provides your next input but should also be appended to the generated output sequence. After generating a sequence of the desired length, return it. Your output should be a list of integers in the range 0 to `vocab_size-1`. The following code shows how it should be called and the output displayed (each time you run it you should get a new random output from the model):

In [ ]:
def sample(model, hprev, seed, n):
    ixes = [seed]
    # TODO: repeat n times:
    #   1. pass the current input (as torch.tensor([x])) and hprev through the model
    #   2. apply softmax to the output y to get probabilities
    #   3. sample the next character index with torch.multinomial
    #   4. append it to ixes and use it as the next input
    return ixes

In [ ]:
seed_token = random.randint(0, vocab_size - 1)
hprev = torch.zeros(hidden_size,)
sample_ix = sample(model, hprev, seed_token, 200)
txt = ''.join(ix_to_char[ix] for ix in sample_ix)
print('----\n%s \n----' % (txt, ))

You could also modify your training loop to output a random sample every 500 iterations.

**To do**: you can introduce a positive "temperature" parameter to control confidence versus diversity when you take a random sample. To do this, you simply divide the logits output by the model by the temperature before you apply softmax. If the temperature is $>1$, you introduce more randomness. If the temperature is $<1$ you make the model more deterministic. Check this is the case by creating multiple random passages with the same start token and hidden state. Do this for various temperature settings.

In [ ]:
# TODO: write a sampling function with a temperature parameter, then generate several
# passages with the same seed token and hidden state for a few temperatures (e.g. 0.1, 0.5, 1.0, 1.5, 2.5)


### Constrained decoding

Constrained decoding (sometimes called "structured output") restricts a language model's output to conform to a schema by masking invalid tokens at each generation step (so you sample only from valid tokens). This is the same idea behind JSON mode, function calling, and libraries like `outlines` and `xgrammar` — at scale the FSM becomes a context-free grammar or regex, but the underlying masking-logits-then-sampling mechanism is identical to what you will implement here.

You must write a constrained decoding function for your Shakespeare model to force its output to match the play-script structure:

```
NAME:
<dialogue>
```

That is: one or more uppercase letters (possibly with spaces, since names like `KING HENRY` are valid), then a colon, then a newline, then free-form text. Generation is seeded with a single uppercase letter and then runs for a fixed number of characters until the desired sequence length is matched. The seed character is fed in as the first input token, forms the first character of the generated output, and satisfies the "≥1 uppercase" requirement, so the FSM begins already inside the name.

#### State Machine

You can think of constrained decoding in terms of a Finite State Machine (FSM). In each state, different characters are permitted:

| State | Meaning                            | Allowed characters     | Transitions                           |
|-------|------------------------------------|------------------------|---------------------------------------|
| S1    | Inside the name (entered via seed) | `A`–`Z`, space, `:`    | `A`–`Z` or space → S1; `:` → S2       |
| S2    | Just emitted the colon             | `\n`                   | → S3                                  |
| S3    | Free generation                    | Anything in vocabulary | → S3                                  |

Once the FSM reaches S3 it stays there for the remainder of the run. A richer FSM could re-enter S1 after a `\n\n` boundary to alternate between speakers - we keep it deliberately simple here.

#### Implementation Steps
1. Define `allowed[state]` as a set (or tensor) of character indices for each state.
2. Choose a fixed generation length `N` and a seed character (an uppercase letter). Initialise the FSM in S1.
3. For each of the `N` steps:
   - Compute logits for the next character.
   - Set logits of characters not in `allowed[current_state]` to `-inf` *before* applying softmax.
   - Sample from the masked distribution.
   - Update the FSM state based on the sampled character.
4. Verify your constrained output actually adheres to the schema. Then compare against unconstrained output of the same length from the same model and seed.

In [ ]:
upper_chars = [c for c in string.ascii_uppercase if c in char_to_ix]

# TODO 1: fill in the allowed character indices for each state
allowed = {
    1: [],  # S1: uppercase letters, space and ':'
    2: [],  # S2: newline only
    3: [],  # S3: anything in the vocabulary
}


def next_state(state, ch):
    """FSM transition function: return the next state after emitting character ch."""
    # TODO 2: implement the transitions from the state machine table above
    #   S1: ':' -> S2, uppercase letter or space -> S1
    #   S2: '\n' -> S3
    #   S3: anything -> S3
    return state


def constrained_sample(model, hprev, seed, n):
    ixes = [seed]
    state = 1
    # TODO 3: for each of the n steps, compute logits, set disallowed logits to -inf,
    # apply softmax, sample, append, and update the FSM state using next_state
    return ixes


def follows_schema(text):
    """Returns True if text matches NAME:\n<dialogue>."""
    return re.fullmatch(r'[A-Z][A-Z ]*:\n.*', text, flags=re.DOTALL) is not None

In [ ]:
N = 200
seed_char = random.choice(upper_chars)
seed_ix = char_to_ix[seed_char]
h0 = torch.zeros(hidden_size,)

constrained_txt = ''.join(ix_to_char[i] for i in constrained_sample(model, h0, seed_ix, N))
unconstrained_txt = ''.join(ix_to_char[i] for i in sample(model, h0, seed_ix, N))

print('===== CONSTRAINED =====')
print(constrained_txt)
print('-> follows schema:', follows_schema(constrained_txt))
print('\n===== UNCONSTRAINED =====')
print(unconstrained_txt)
print('-> follows schema:', follows_schema(unconstrained_txt))

### Optional challenge: generating baby names

We will now use a different dataset to generate novel baby names.

Different to the previous exercise, these sequences have an end: the end of the name. So, you will need to increase the vocabulary by one with the last value meaning "end of sequence" (EOS). At each iteration you will process one baby name. You will pass the first character of the name as the initial input and hidden state as all zeroes. The targets will be the name starting from the second character with the extra EOS character index appended to the end. i.e. If the name is "John" then the inputs will be token indices for "John" and the targets will be the token indices for "ohn`<EOS>`".

The model can be trained in exactly the same way. When sampling from the trained model, you do not need to provide a desired sequence length. Instead, you stop when the next character is EOS.

You can download the dataset with one name per line here:

In [ ]:
fname = download('https://www-users.york.ac.uk/~waps101/PADL/babies.txt')
names = [line.strip() for line in open(fname, 'r', encoding='utf-8', errors='replace') if line.strip()]
print(len(names), 'names, e.g.', names[:10])

In [ ]:
# TODO (optional):
# 1. Build a character vocabulary from the names plus one extra EOS index
# 2. Create a NEW RNN (e.g. name_model) so you don't overwrite the Shakespeare model
# 3. Train one name per iteration: inputs = name, targets = name[1:] + [EOS], hidden state = zeros
# 4. Sample names until EOS is generated, and check which ones are not in `names`


Can your model generate new names that weren't in the training set?

## Zero-shot image classification with CLIP

CLIP is a vision and language model. It uses a Vision transformer to embed images and transformer text encoder to embed captions. The embedding space is shared: if an image and a caption embed to similar vectors then their meaning/contents should be similar. Because it is pretrained on such a large dataset, it can be used as a zero-shot classifier. i.e. you can use it to classify anything you can describe with a caption. You will use it to classify cat versus dog images.

In [ ]:
%pip install -q transformers

In [ ]:
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import matplotlib.pyplot as plt

In [ ]:
# Named clip_model so it doesn't overwrite the Shakespeare `model` above
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
tokenizer = processor.tokenizer
clip_model.eval()


def get_image_embeddings(imgs):
    """Image(s) -> N x 512 CLIP embeddings."""
    inputs = processor(images=imgs, return_tensors="pt")
    with torch.no_grad():
        out = clip_model.vision_model(pixel_values=inputs['pixel_values'])
        return clip_model.visual_projection(out.pooler_output)


def get_text_embeddings(texts):
    """List of captions -> N x 512 CLIP embeddings."""
    inputs = tokenizer(texts, padding=True, return_tensors="pt")
    with torch.no_grad():
        out = clip_model.text_model(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'])
        return clip_model.text_projection(out.pooler_output)

In [ ]:
zname = download('https://www-users.york.ac.uk/~waps101/PADL/cats_and_dogs.zip')
with zipfile.ZipFile(zname) as z:
    z.extractall('.')

In [ ]:
filenames = sorted(f for f in os.listdir('cats_and_dogs') if f.lower().endswith(('.jpg', '.jpeg', '.png')))
images = [Image.open(os.path.join('cats_and_dogs', f)).convert('RGB') for f in filenames]
print(len(images), 'images loaded')

You can get the embedding for an image like:

In [ ]:
i = 0  # index of the image to embed
image_embedding = get_image_embeddings(images[i])
print(image_embedding.shape)

You can get the embedding for a text prompt like:

In [ ]:
text_embedding = get_text_embeddings(["A photo of a dog"])
print(text_embedding.shape)

The two embeddings are PyTorch tensors of shape $1\times 512$. You can pass more than one image or caption at a time and then the first dimension will be the batch size.

Similarity between embeddings is measured by the dot product between their embedding vectors.

**To do**: Can you classify the images as dogs and cats? Display the images along with their classification.

Try some other random images. How discriminative is CLIP? How about the MNIST digits? These are probably quite out-of-domain for its training images.

In [ ]:
# TODO:
# 1. Embed captions such as 'a photo of a cat' and 'a photo of a dog'
# 2. Embed all the images (you can pass a list of images to get_image_embeddings)
# 3. Compute dot products between every image and every caption; the highest score is the prediction
#    (hint: normalise the embeddings first so the dot product is a cosine similarity)
# 4. Display each image with its predicted label using matplotlib


In [ ]:
# TODO: try other images / labels, and MNIST digits
# (hint: from torchvision.datasets import MNIST; convert each digit image with .convert('RGB'))
